In [1]:
import pandas as pd 
import re 
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 
from difflib import SequenceMatcher

In [ ]:
df = pd.read_csv("fakenews.csv")
#missimng text values filled
df["headline"] = df["headline"].fillna("") 
df["body_text"] = df["body_text"].fillna("")

In [ ]:
# Function to clean text
def clean_text(text):
     text = text.lower() 
     text = re.sub(r'[^a-z0-9\s]', '', text) 
     text = re.sub(r'\s+', ' ', text).strip() 
     return text

In [ ]:
# Cleaned headlines and body text
df["clean_headline"] = df["headline"].apply(clean_text) 
df["clean_body"] = df["body_text"].apply(clean_text)

In [5]:
# Detect exact duplicate headlines
df["exact_duplicate"] = df.duplicated( subset=["clean_headline"], keep="first" )

In [6]:
# Combine headline and body text 
df["combined_text"] = ( df["clean_headline"] + " " + df["clean_body"] )

In [7]:
# TF-IDF vectorization 
vectorizer = TfidfVectorizer(stop_words="english") 
tfidf_matrix = vectorizer.fit_transform(df["combined_text"])

In [9]:
# Calculate cosine similarity 
similarity_matrix = cosine_similarity(tfidf_matrix)

In [11]:
threshold = 0.80
near_duplicates = []

# Calculate similarities in batches instead of creating
# one huge similarity matrix
batch_size = 500

for start in range(0, len(df), batch_size):
    end = min(start + batch_size, len(df))

    batch_similarity = cosine_similarity(
        tfidf_matrix[start:end],
        tfidf_matrix
    )

    for row_idx in range(end - start):
        i = start + row_idx

        for j in range(i + 1, len(df)):
            similarity = batch_similarity[row_idx, j]

            if similarity >= threshold:
                headline_similarity = SequenceMatcher(
                    None,
                    df.iloc[i]["clean_headline"],
                    df.iloc[j]["clean_headline"]
                ).ratio()

                near_duplicates.append({
                    "news_id_1": df.iloc[i]["news_id"],
                    "news_id_2": df.iloc[j]["news_id"],
                    "cosine_similarity": round(similarity, 3),
                    "headline_similarity": round(
                        headline_similarity, 3
                    )
                })

In [12]:
print("Near-Duplicate News Articles:")

if near_duplicates:
    print(pd.DataFrame(near_duplicates))
else:
    print("No near-duplicate articles found.")

Near-Duplicate News Articles:
       news_id_1  news_id_2  cosine_similarity  headline_similarity
0              1         20                1.0                  1.0
1              1        158                1.0                  1.0
2              1        211                1.0                  1.0
3              1        300                1.0                  1.0
4              1        303                1.0                  1.0
...          ...        ...                ...                  ...
24329       2184       2197                1.0                  1.0
24330       2186       2190                1.0                  1.0
24331       2193       2199                1.0                  1.0
24332       2206       2207                1.0                  1.0
24333       2209       2210                1.0                  1.0

[24334 rows x 4 columns]


In [13]:
# Display exact duplicate headline
print("\nExact Duplicate Headlines:") 
print(df.loc[ df["exact_duplicate"], ["news_id", "headline"] ])


Exact Duplicate Headlines:
      news_id                                   headline
5           6  Earthquake hits Chittagong, panic spreads
6           7             Global economy faces recession
7           8             Global economy faces recession
8           9  Earthquake hits Chittagong, panic spreads
10         11       New curriculum introduced in schools
...       ...                                        ...
2207     2208              Bangladesh wins the World Cup
2208     2209       New curriculum introduced in schools
2209     2210       New curriculum introduced in schools
2210     2211    Dengue situation worsens in the country
2211     2212           New metro rail launched in Dhaka

[2202 rows x 2 columns]


In [14]:
# Remove exact duplicate headlines 
# Keep the first occurrence 
cleaned_df = df.drop_duplicates( subset=["clean_headline"], keep="first" ).copy()

In [15]:
# Remove temporary columns
cleaned_df = cleaned_df.drop(columns=[ "clean_headline", "clean_body", "combined_text", "exact_duplicate" ])

In [16]:
# Save cleaned dataset 
cleaned_df.to_csv("fakenews_cleaned.csv", index=False) 
print("\nOriginal dataset size:", len(df)) 
print("Cleaned dataset size:", len(cleaned_df)) 
print("Duplicate headlines removed:", len(df) - len(cleaned_df)) 
print("\nCleaned dataset saved as fakenews_cleaned.csv")


Original dataset size: 2212
Cleaned dataset size: 10
Duplicate headlines removed: 2202

Cleaned dataset saved as fakenews_cleaned.csv
